In [1]:
import pandas as pd
import numpy as np
import random
from itertools import product

# 设置随机种子以确保可重复性
random.seed(42)
np.random.seed(42)

def create_negative_samples(positive_df, gene_list=None, allow_self_loop=False):
    """
    为基因调控网络构建负样本
    
    参数:
    positive_df: pandas DataFrame, 正样本数据，至少包含两列表示基因对
    gene_list: list, 所有基因的列表（可选）。如果为None，则从正样本中提取所有基因
    allow_self_loop: bool, 是否允许自环（基因对自身）。默认为False
    
    返回:
    negative_df: pandas DataFrame, 负样本数据
    """
    
    # 确保正样本数据有两列
    if len(positive_df.columns) < 2:
        raise ValueError("正样本DataFrame至少需要两列来表示基因对")
    
    # 获取基因对列名
    gene_col1, gene_col2 = positive_df.columns[:2]
    
    # 如果没有提供基因列表，从正样本中提取所有唯一基因
    if gene_list is None:
        tf_list = list(set(positive_df[gene_col1].tolist()))
        gene_list = list(set((positive_df[gene_col2].tolist())))

    # 创建所有可能的基因对（包括自环或不包括，根据参数）
    if allow_self_loop:
        all_possible_pairs = list(product(gene_list, repeat=2))
    else:
        all_possible_pairs = [(g1, g2) for g1 in tf_list for g2 in gene_list if g1 != g2]
    
    # 将正样本转换为集合以便快速查找
    positive_pairs = set(positive_df.apply(lambda row: tuple(row[:2]), axis=1))
    
    # 获取所有非正样本的基因对
    negative_candidates = [pair for pair in all_possible_pairs if pair not in positive_pairs]
    
    # 检查是否有足够的负样本候选
    n_positive = len(positive_df)
    if len(negative_candidates) < n_positive:
        print(f"警告: 负样本候选数量({len(negative_candidates)})少于正样本数量({n_positive})")
        print("将使用所有可用的负样本候选")
        n_samples = min(n_positive, len(negative_candidates))
    else:
        n_samples = n_positive
    
    # 随机抽取负样本
    selected_negative_pairs = random.sample(negative_candidates, n_samples)
    
    # 创建负样本DataFrame
    negative_df = pd.DataFrame(selected_negative_pairs, columns=[gene_col1, gene_col2])
    
    # 如果正样本有其他列，可以在负样本中添加对应的占位符列
    if len(positive_df.columns) > 2:
        for col in positive_df.columns[2:]:
            # 对于数值列，可以填充0；对于分类列，可以填充空值或特定值
            if pd.api.types.is_numeric_dtype(positive_df[col]):
                negative_df[col] = 0
            else:
                negative_df[col] = np.nan
    
    return negative_df

# 示例用法
if __name__ == "__main__":   
    
    pos = pd.read_csv("BL--network.csv")
    #del pos["Score"]
    print(pos)
    pos["label"] = 1
    pos.columns=["TF","Target","label"]
    # 方法1: 使用从正样本中提取的基因列表
    negative_df1 = create_negative_samples(pos)
    negative_df1["label"]=0
    negative_df1.columns=["TF","Target","label"]
    print(len(set(list(negative_df1["TF"]))))
    print(len(set(list(pos["TF"]))))
    print((set(list(pos["TF"]))))
    all=pd.concat([pos,negative_df1])   
    print(all)
    all.to_csv("AllPair.csv",index=False)

         Gene1   Gene2
0        APBB1  NOTCH1
1         BMI1    CDK1
2         BMI1   IKZF1
3         E2F4    MYCN
4         E2F4   RAD51
...        ...     ...
1955     ZFP36   CUL4A
1956     ZFP36   DUSP1
1957    ZFP592    NFE2
1958  ZKSCAN17  JARID2
1959   ZSCAN21     PML

[1960 rows x 2 columns]
147
147
{'EGR1', 'HIST1H1C', 'RUVBL2', 'XRCC5', 'MIS18BP1', 'NFATC2', 'MEIS1', 'GTF2E1', 'PLAG1', 'BTG2', 'RORC', 'SPO11', 'ABL1', 'SP4', 'BMI1', 'LZTS1', 'PGR', 'NDN', 'IRF6', 'ESR1', 'FOS', 'IRF8', 'VHL', 'IRF5', 'RELB', 'GFI1B', 'TEAD3', 'TOPORS', 'HDAC9', 'SMARCA2', 'ZFP592', 'PWP1', 'JARID2', 'HDGF', 'XAB2', 'REL', 'SATB1', 'STAT3', 'HDAC10', 'POLR2B', 'RAI14', 'EPAS1', 'GATA1', 'HOXA9', 'ZKSCAN17', 'MCM4', 'ZDHHC6', 'SPI1', 'E2F4', 'ID2', 'RNF4', 'XPC', 'POLR1B', 'SNAPC3', 'AHR', 'ETS1', 'DCP1A', 'FOXO1', 'GTF2F1', 'NRF1', 'HDAC2', 'IKZF1', 'RB1', 'NFE2L2', 'NFKBIA', 'H1F0', 'GTF3C3', 'YBX1', 'RUNX1T1', 'HDAC11', 'MCM5', 'TRIB3', 'GATA2', 'MYCN', 'TCF19', 'APBB1', 'HNF4A', 'SMAD2', 'P

In [2]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
a

,CDK19,NXPE2,CNOT7,EMILIN1,ORC6,ETF1,NGLY1,MCM3,ZFP655,CCNB1,...,SMIM1,GLIS2,REL,H1F0,RPP40,LYZ2,SLC20A1,TBL3,XPNPEP1,UTP4
0,2.129140,3.238239,2.954035,8.373768,0.000000,7.656687,1.426148,9.319204,2.129140,2.129140,...,0.000000,1.426148,4.409214,3.475564,6.699389,9.835766,0.000000,2.129140,2.954035,7.960179
1,7.435393,3.683348,8.754396,0.000000,2.855035,2.725019,1.168027,9.032075,7.583439,1.168027,...,0.699126,0.000000,0.699126,7.304942,0.000000,10.257798,1.521334,2.041805,0.000000,1.521334
2,9.717998,3.453502,8.990668,8.567140,2.579528,3.453502,0.000000,5.517180,2.579528,0.000000,...,0.000000,0.000000,8.274268,7.936017,8.528769,10.241409,3.453502,3.453502,2.579528,2.579528
3,0.000000,1.693409,9.313805,0.000000,1.226829,8.478506,0.532906,2.328136,9.386882,2.193684,...,0.532906,6.978599,0.000000,0.532906,9.521034,10.943163,9.595419,9.201131,1.226829,0.532906
4,1.172368,1.172368,8.243978,0.000000,1.810495,1.810495,0.000000,9.300156,5.711669,1.172368,...,1.810495,0.000000,1.810495,3.463426,0.000000,9.077188,1.172368,9.113563,1.172368,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066,2.539502,3.173629,4.552425,1.381077,7.118920,7.102168,5.710064,8.556440,5.795905,8.525363,...,5.710064,4.221173,1.381077,2.073565,0.000000,3.612634,8.556440,4.901267,7.973392,5.665150
1067,2.745032,8.274210,7.954110,1.714408,9.110043,8.655120,4.466299,9.646159,6.491583,8.228299,...,8.194989,3.419118,0.651136,5.394690,5.114945,3.494068,7.811205,7.602641,8.753737,6.588828
1068,2.317134,7.866449,7.532334,6.995304,8.545623,8.579401,6.235670,9.108626,6.395904,6.944367,...,7.332902,1.218970,0.528673,4.938017,4.999193,9.486979,6.083912,6.980211,8.644673,7.948872
1069,3.786364,2.260608,6.431598,8.267474,4.284343,6.209351,6.298927,8.738493,5.596911,0.559729,...,0.961982,0.000000,0.000000,0.559729,0.559729,10.041298,1.534045,1.534045,7.132922,8.341018


In [4]:
#转为大写
import pandas as pd
a=pd.read_csv("pathway_mHSC-E.csv")
col = list(a.columns)
cols = [i.upper() for i in col]
a.columns=cols
a.to_csv("pathway_mHSC-E.csv",index=False)

In [5]:
import pandas as pd
a=pd.read_csv("Target.csv")
column_names = list(a["Target"])
uppercase_names = [name.upper() for name in column_names]
a["Target"] = uppercase_names
a.to_csv("AllPair.csv",index=False)

KeyError: 'Target'

In [6]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
column_names = list(a.columns)
uppercase_names = [name.upper() for name in column_names]
a.columns = uppercase_names
a.to_csv("ExpressionData.csv",index=False)